# Spark and Databricks

The previous chapters were about getting the right answer. This chapter is
about getting it efficiently: understanding what a workload actually does on
the cluster, finding where its time and money go, and fixing the expensive
parts. This first notebook looks at what Databricks changes about how Spark
runs, and at the tools it provides for watching workloads. The other two put
them to use:
{doc}`Using Apache Spark with Delta Lake <02_using_apache_spark_with_delta_lake>`
covers the storage layer that most optimizations rely on, and
{doc}`Optimizing Apache Spark <03_optimizing_apache_spark>` the techniques
themselves.

In [ ]:
from pyspark.sql import Observation
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

spark.sql("DROP TABLE IF EXISTS workspace.default.monitor01_large_orders");

## What Databricks Changes About Spark

{doc}`What Is Apache Spark? <../01_introduction_to_apache_spark/01_apache_spark_runtime_architecture/01_what_is_apache_spark>`
listed what Databricks adds to open-source Spark: managed compute, the
Databricks Runtime, Delta Lake and Unity Catalog. For performance, four of
these additions matter most:

- **Photon**, a vectorised query engine written in C++, executes most
  DataFrame and SQL operations instead of Spark's JVM engine. Plans on
  Databricks show `Photon...` operators where it is used.
- **Delta Lake** stores per-file statistics and supports clustering and
  compaction, which let queries skip most of a large table.
- **Serverless compute** takes most tuning decisions out of your hands, as
  shown below.
- **Predictive optimization** maintains Unity Catalog managed tables in the
  background, as shown in
  {doc}`Using Apache Spark with Delta Lake <02_using_apache_spark_with_delta_lake>`.

On serverless compute, settings that you would tune on a classic cluster are
managed for you. The number of shuffle partitions is `auto`, chosen per
query from the data size:

In [2]:
spark.conf.get("spark.sql.shuffle.partitions")

'auto'

And Adaptive Query Execution is always on. Trying to change such a setting
fails:

In [3]:
try:
    spark.conf.set("spark.sql.adaptive.enabled", "false")
except Exception as e:
    print(type(e).__name__, "-", str(e).splitlines()[0])

AnalysisException - [CONFIG_NOT_AVAILABLE.WITHOUT_SUGGESTION] Configuration spark.sql.adaptive.enabled is not available.  SQLSTATE: 42K0I


The consequence for this chapter: on serverless compute, optimizing is less
about Spark configuration and more about how data is laid out and how
queries are written. Those are also the changes that help most on classic
compute.

## Reading a Query Plan

The first monitoring tool needs no UI at all. `explain()` prints the
physical plan Spark will run. Here is a query over `samples.tpch.lineitem`,
a 30-million-row table of order lines from the TPC-H benchmark, that counts
the lines shipped on one day per return flag:

In [4]:
lineitem = spark.read.table("samples.tpch.lineitem")

returns_on_new_year = (
    lineitem
    .filter(F.col("l_shipdate") == "1995-01-01")
    .groupBy("l_returnflag")
    .agg(F.count("*").alias("lines"))
)

returns_on_new_year.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[l_returnflag#63812], functions=[finalmerge_count(merge count#63830L) AS count(1)#63822L])
         +- PhotonShuffleExchangeSource false
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#45812]
               +- PhotonShuffleExchangeSink hashpartitioning(l_returnflag#63812, 1024)
                  +- PhotonGroupingAgg(keys=[l_returnflag#63812], functions=[partial_count(1) AS count#63830L])
                     +- PhotonProject [l_returnflag#63812]
                        +- PhotonScan parquet samples.tpch.lineitem[l_returnflag#63812,l_shipdate#63814] DataFilters: [isnotnull(l_shipdate#63814), (l_shipdate#63814 = 1995-01-01)], DictionaryFilters: [(l_shipdate#63814 = 1995-01-01)], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://system-tables-prod-us-east-2-uc-metastore-bucket/metastore/b..., OptionalDataF

:::::{dropdown} Example: Reading the Plan from the Bottom Up
:open:
:color: primary
:icon: pencil
:class-body: example-body

A physical plan is a tree, printed with the first operator to run at the
bottom. Reading the plan above upwards:

**`PhotonScan parquet samples.tpch.lineitem[l_returnflag, l_shipdate]`.**
Only the two columns the query uses are read, and the `DataFilters` show
that the date filter is applied inside the scan.

**`PhotonGroupingAgg(... partial_count(1) ...)`.** Each task counts its own
rows per return flag before anything leaves the executor.

**`PhotonShuffleExchangeSink hashpartitioning(l_returnflag, 1024)`.** The
partial counts are shuffled so that all counts for one flag meet in one
task. The 1024 is the initial number of shuffle partitions; AQE reduces it
at run time based on the actual data size.

**`PhotonGroupingAgg(... finalmerge_count ...)`.** The partial counts are
added up into the final result.

The sections after the plan add Databricks-specific information. `Photon
Explanation` says whether every operator runs in Photon, and `Optimizer
Statistics` lists tables without statistics and suggests running
`ANALYZE TABLE`. The whole query has a single shuffle, which is the
minimum for an aggregation.
:::::

`explain(mode="formatted")` prints the same plan as a numbered list of
operators with their details, which is easier to read for large plans. How
to interpret the operators is covered in
{doc}`Distributed Systems Programming Fundamentals <../01_introduction_to_apache_spark/03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`.

A plan shows what Spark **intends** to do. To see what it actually did, how
many rows each operator produced, how long it took and where the data was
skewed, you need the query profile or the Spark UI.

## The Query Profile and the Spark UI

Which UI you use depends on the compute:

- **Serverless compute and SQL warehouses** have a **query profile**. Open
  it from the query history in the Databricks UI, or from the link below a
  notebook cell's output. It shows the executed plan as a graph, annotated
  with rows, time, memory and spill per operator, and summary metrics such
  as bytes read, files pruned and time spent in Photon.
- **Classic compute** has the **Spark UI**, available from the compute
  page. Its **Jobs** and **Stages** tabs show the jobs, stages and tasks
  described in
  {doc}`Distributed Systems Programming Fundamentals <../01_introduction_to_apache_spark/03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`,
  with the distribution of task durations per stage. The **SQL / DataFrame**
  tab shows executed plans with per-operator metrics.

What to look for is the same in both:

| Symptom | Likely cause |
| --- | --- |
| Most time spent in scan operators, many bytes read | The query reads more than it needs: missing filters, no clustering, too many small files |
| Large shuffle reads and writes | Joins or aggregations that move a lot of data; candidates for broadcasting or pre-aggregation |
| One task much slower than the others in a stage | Data skew |
| Spill to disk | Too little memory per task for the data being sorted, joined or aggregated |
| Operators not running in Photon | Python UDFs or unsupported functions forcing the slower engine |

{doc}`Optimizing Apache Spark <03_optimizing_apache_spark>` addresses each
of these.

## Query History in System Tables

Looking at one query at a time does not answer questions like "which of my
queries read the most data last week?". Databricks records every query run
on serverless compute and SQL warehouses in the **system table**
`system.query.history`, with the same metrics as the query profile. As a
Unity Catalog table, it can be queried like any other. It has a column for
each metric:

In [5]:
history = spark.read.table("system.query.history")

[column for column in history.columns if column.endswith(("_ms", "_bytes", "_files", "_rows"))]

['total_duration_ms',
 'waiting_for_compute_duration_ms',
 'waiting_at_capacity_duration_ms',
 'execution_duration_ms',
 'compilation_duration_ms',
 'total_task_duration_ms',
 'result_fetch_duration_ms',
 'pruned_files',
 'read_files',
 'read_rows',
 'produced_rows',
 'read_bytes',
 'spilled_local_bytes',
 'written_bytes',
 'shuffle_read_bytes',
 'written_rows',
 'written_files',
 'pruned_files_bytes',
 'read_files_bytes']

Queries sent through Databricks Connect appear with `client_application` set
to `SPARK_CONNECT`, and their `statement_text` is the code of the notebook
cell that ran them. Here are the queries over the TPC-H sample tables that
read the most data in the last week. Your output lists whichever of the
course's queries you have run. Records appear in the system table a few
minutes after a query finishes, so very recent queries may be missing.

In [6]:
(
    history
    .filter(F.col("client_application") == "SPARK_CONNECT")
    .filter(F.col("start_time") >= F.current_timestamp() - F.expr("INTERVAL 7 DAYS"))
    .filter(F.col("statement_text").contains("samples.tpch"))
    .select(
        "statement_type",
        (F.col("total_duration_ms") / 1000).alias("seconds"),
        "read_files",
        "pruned_files",
        F.round(F.col("read_bytes") / 1024**2).alias("read_mb"),
        F.round(F.col("shuffle_read_bytes") / 1024**2).alias("shuffle_mb"),
        "read_rows",
    )
    .orderBy(F.desc("read_mb"))
    .show(10)
)

+--------------+-------+----------+------------+-------+----------+---------+
|statement_type|seconds|read_files|pruned_files|read_mb|shuffle_mb|read_rows|
+--------------+-------+----------+------------+-------+----------+---------+
|        CREATE| 16.936|        20|           0| 1043.0|       0.0| 59999590|
|        CREATE|  10.32|        10|           0|  721.0|       0.0| 29999795|
|        SELECT|   0.87|        10|           0|   46.0|       0.0|    12357|
|      DESCRIBE|   0.63|         0|           0|    0.0|       0.0|        0|
|          SHOW|  4.748|         0|           0|    0.0|       0.0|        0|
+--------------+-------+----------+------------+-------+----------+---------+



Two columns deserve a closer look. `read_files` and `pruned_files` show how
well a query used **data skipping**: a query on a well-laid-out table prunes
most files and reads few. `shuffle_mb` shows how much data crossed the
network between stages. Aggregated over all queries, the same table answers
questions like which tables are scanned most often, or which users and jobs
run the most expensive queries.

## Cost in System Tables

The other half of "efficient" is money. Databricks bills compute in
**Databricks Units (DBUs)**, and `system.billing.usage` records every unit
consumed, by product, SKU, compute resource, job and user. Here is the
usage of this workspace over the last 30 days, by product:

In [7]:
(
    spark.read.table("system.billing.usage")
    .filter(F.col("usage_date") >= F.date_sub(F.current_date(), 30))
    .groupBy("billing_origin_product")
    .agg(F.round(F.sum("usage_quantity"), 2).alias("dbus"))
    .orderBy(F.desc("dbus"))
    .show()
)

+----------------------+----+
|billing_origin_product|dbus|
+----------------------+----+
|                  JOBS|9.40|
|                  APPS|8.00|
|       DEFAULT_STORAGE|7.86|
|                   SQL|4.48|
|  PREDICTIVE_OPTIMI...|1.38|
|                 GENIE|0.46|
|            NETWORKING|0.00|
+----------------------+----+



Joining usage with `system.billing.list_prices` turns DBUs into currency,
and grouping by `usage_metadata.job_id` or `identity_metadata.run_as` shows
which jobs or users drive the cost. Before optimizing a workload, check
that it actually matters: a ten-times speed-up of a job that costs a few
cents a month is not worth an afternoon.

Other system tables are useful for monitoring too, for example
`system.lakeflow` for job runs, `system.compute` for classic clusters, and
`system.access.table_lineage` for which jobs read and write which tables.

## Observing Metrics in Your Own Code

Sometimes you want metrics about the data itself, such as how many rows a
step wrote or how many had a missing value, for logging or data-quality
checks. Calling `count()` for each one would re-run the whole plan every
time. An **observation** computes such metrics as a side effect of an
action that runs anyway:

In [8]:
transactions = spark.read.table("samples.bakehouse.sales_transactions")

write_stats = Observation("large_orders")

large_orders = (
    transactions
    .filter(F.col("quantity") >= 30)
    .observe(
        write_stats,
        F.count(F.lit(1)).alias("rows"),
        F.sum("totalPrice").alias("revenue"),
        F.max("quantity").alias("largest_quantity"),
    )
)

large_orders.write.mode("overwrite").saveAsTable("workspace.default.monitor01_large_orders")

write_stats.get

{'rows': 66, 'revenue': 8352, 'largest_quantity': 60}

The metrics were collected while the table was written, in the same job, so
they cost almost nothing extra. The 66 rows are the same large orders counted
in {doc}`DataFrame API Basics <../02_developing_applications_with_apache_spark/01_dataframe_api_basics>`,
which needed a separate `count()` there. Streaming queries report similar metrics in
every progress report, as shown in
{doc}`Spark Structured Streaming <../03_stream_processing_and_analysis/02_spark_structured_streaming>`.

In [9]:
spark.sql("DROP TABLE IF EXISTS workspace.default.monitor01_large_orders");

Next, {doc}`Using Apache Spark with Delta Lake <02_using_apache_spark_with_delta_lake>`
looks at the storage layer underneath every table in this course.

## Resources

- [Query profile (Databricks)](https://docs.databricks.com/aws/en/sql/user/queries/query-profile)
- [Query history system table reference (Databricks)](https://docs.databricks.com/aws/en/admin/system-tables/query-history)
- [Billable usage system table reference (Databricks)](https://docs.databricks.com/aws/en/admin/system-tables/billing)
- [Debugging with the Spark UI (Databricks)](https://docs.databricks.com/aws/en/compute/troubleshooting/debugging-spark-ui)
- [Web UI (Apache Spark)](https://spark.apache.org/docs/latest/web-ui.html)